# Explainability and App Preparation

This notebook prepares deployment-facing artifacts: feature importance, model summaries, and example rows for the Streamlit app.


In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)


/Users/paco/Documents/Git Projects/network-intrusion-detection-system


In [2]:
import json

import joblib
import pandas as pd
from sklearn.inspection import permutation_importance

from src.config import CLEANED_DIR, FEATURE_COLUMNS, MODELS_DIR, REPORTS_DIR, ensure_project_dirs

ensure_project_dirs()
train_df = pd.read_csv(CLEANED_DIR / "train_cleaned.csv")
test_df = pd.read_csv(CLEANED_DIR / "test_cleaned.csv")
binary_model = joblib.load(MODELS_DIR / "binary_intrusion_model.pkl")
multiclass_model = joblib.load(MODELS_DIR / "multiclass_attack_model.pkl")


## Feature Importance


In [3]:
sample = test_df.sample(n=min(2500, len(test_df)), random_state=42)
X_sample = sample[FEATURE_COLUMNS]
y_binary_sample = sample["binary_label"]
y_multiclass_sample = sample["attack_category"]


def permutation_importance_frame(model, X, y, scoring):
    result = permutation_importance(
        model,
        X,
        y,
        scoring=scoring,
        n_repeats=5,
        random_state=42,
        n_jobs=-1,
    )
    importance = pd.DataFrame(
        {
            "feature": X.columns,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
        }
    ).sort_values("importance_mean", ascending=False)
    return importance

binary_importance = permutation_importance_frame(binary_model, X_sample, y_binary_sample, scoring="f1")
multiclass_importance = permutation_importance_frame(multiclass_model, X_sample, y_multiclass_sample, scoring="f1_weighted")

display(binary_importance.head(20))
display(multiclass_importance.head(20))

binary_importance.to_csv(REPORTS_DIR / "binary_feature_importance.csv", index=False)
multiclass_importance.to_csv(REPORTS_DIR / "multiclass_feature_importance.csv", index=False)


,feature,importance_mean,importance_std
4,src_bytes,8.162446e-02,0.005038
5,dst_bytes,2.763951e-02,0.002521
39,dst_host_rerror_rate,2.740314e-02,0.001593
0,duration,2.203264e-02,0.001371
32,dst_host_srv_count,1.423697e-02,0.002081
37,dst_host_serror_rate,9.309094e-03,0.001035
35,dst_host_same_src_port_rate,7.722407e-03,0.001682
9,hot,6.848306e-03,0.000414
36,dst_host_srv_diff_host_rate,6.718277e-03,0.000989
38,dst_host_srv_serror_rate,3.099432e-03,0.000862


,feature,importance_mean,importance_std
4,src_bytes,0.262780,0.004172
2,service,0.075749,0.001853
35,dst_host_same_src_port_rate,0.041234,0.002257
28,same_srv_rate,0.023782,0.002739
29,diff_srv_rate,0.021277,0.001634
5,dst_bytes,0.015911,0.001680
9,hot,0.011839,0.001271
37,dst_host_serror_rate,0.010791,0.002622
21,is_guest_login,0.010345,0.000366
10,num_failed_logins,0.009460,0.000982


## Example App Inputs


In [4]:
examples = pd.concat(
    [
        test_df[test_df["binary_label"] == 0].head(5),
        test_df[test_df["binary_label"] == 1].head(5),
    ],
    ignore_index=True,
)

examples[FEATURE_COLUMNS + ["binary_label", "attack_category", "attack_type"]].to_csv(
    REPORTS_DIR / "example_connections.csv",
    index=False,
)

display(examples[FEATURE_COLUMNS + ["binary_label", "attack_category", "attack_type"]].head(10))


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,binary_label,attack_category,attack_type
0,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.04,0.61,0.02,0.00,0.00,0.00,0.00,0,normal,normal
1,0,tcp,http,SF,267,14515,0,0,0,0,...,0.00,0.01,0.03,0.01,0.00,0.00,0.00,0,normal,normal
2,0,tcp,smtp,SF,1022,387,0,0,0,0,...,0.72,0.00,0.00,0.00,0.00,0.72,0.04,0,normal,normal
3,0,tcp,http,SF,327,467,0,0,0,0,...,0.00,0.01,0.03,0.00,0.00,0.00,0.00,0,normal,normal
4,0,tcp,smtp,SF,616,330,0,0,0,0,...,0.03,0.00,0.00,0.00,0.00,0.33,0.00,0,normal,normal
5,0,tcp,private,REJ,0,0,0,0,0,0,...,0.06,0.00,0.00,0.00,0.00,1.00,1.00,1,dos,neptune
6,0,tcp,private,REJ,0,0,0,0,0,0,...,0.06,0.00,0.00,0.00,0.00,1.00,1.00,1,dos,neptune
7,0,icmp,eco_i,SF,20,0,0,0,0,0,...,0.00,1.00,0.28,0.00,0.00,0.00,0.00,1,probe,saint
8,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.17,0.03,0.02,0.00,0.00,0.83,0.71,1,probe,mscan
9,0,tcp,telnet,SF,129,174,0,0,0,0,...,0.00,0.00,0.00,0.01,0.01,0.02,0.02,1,r2l,guess_passwd


## Deployment Notes


In [5]:
deployment_summary = {
    "binary_model": "Models/binary_intrusion_model.pkl",
    "multiclass_model": "Models/multiclass_attack_model.pkl",
    "feature_count": len(FEATURE_COLUMNS),
    "supported_outputs": {
        "binary": ["normal", "attack"],
        "multiclass": ["normal", "dos", "probe", "r2l", "u2r"],
    },
}

with open(REPORTS_DIR / "deployment_summary.json", "w", encoding="utf-8") as file:
    json.dump(deployment_summary, file, indent=2)

deployment_summary


{'binary_model': 'Models/binary_intrusion_model.pkl',
 'multiclass_model': 'Models/multiclass_attack_model.pkl',
 'feature_count': 41,
 'supported_outputs': {'binary': ['normal', 'attack'],
  'multiclass': ['normal', 'dos', 'probe', 'r2l', 'u2r']}}